# Notebook 4 — Image Experiments: CFM vs EFM

**Goal:** Compare Vanilla Conditional Flow Matching (CFM) and Empirical Flow Matching (EFM) on MNIST / Fashion-MNIST.

We replicate the key finding of the paper: regressing against a more deterministic target (EFM) does **not** hurt generalization and can even improve it.

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/YentlCollin/ClosedFlowMatching.git 2>/dev/null || true
    os.chdir('ClosedFlowMatching')
    !pip install -q -r requirements.txt
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, '..')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from tqdm import trange

from src.data.images import get_image_dataloader, extract_all_images
from src.models.unet import SmallUNet
from src.flow_matching.sampler import ode_sample
from src.flow_matching.efm import compute_efm_target
from src.metrics.evaluation import nearest_neighbor_distance

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 4.1 Load data

In [ ]:
DATASET = 'fashion_mnist'
IMAGE_SIZE = 28
BATCH_SIZE = 128
N_TRAIN = 10000  # subset for faster training on Colab

train_loader = get_image_dataloader(DATASET, BATCH_SIZE, IMAGE_SIZE, train=True, n_samples=N_TRAIN)
test_loader = get_image_dataloader(DATASET, BATCH_SIZE, IMAGE_SIZE, train=False)

all_train_images = extract_all_images(train_loader).to(device)
train_flat = all_train_images.view(len(all_train_images), -1)

print(f'Training images: {all_train_images.shape}')
print(f'Flat dimension:  {train_flat.shape[1]}')

## 4.2 Training helpers

We define generic training loops for both CFM and EFM on images.

In [ ]:
def train_cfm_image(model, train_data, n_epochs=20, batch_size=128, lr=2e-4, device='cpu'):
    """Train a UNet with vanilla CFM on image data."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []

    n = len(train_data)
    steps_per_epoch = n // batch_size

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        perm = torch.randperm(n)
        for i in range(steps_per_epoch):
            idx = perm[i * batch_size:(i + 1) * batch_size]
            x1 = train_data[idx].to(device)

            t = torch.rand(batch_size, device=device)
            x0 = torch.randn_like(x1)
            t_expand = t[:, None, None, None]
            x_t = (1.0 - t_expand) * x0 + t_expand * x1

            target = x1 - x0
            pred = model(x_t, t)
            loss = ((pred - target) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg = epoch_loss / steps_per_epoch
        losses.append(avg)
        print(f'  Epoch {epoch+1}/{n_epochs} — loss: {avg:.4f}')
    return losses


def train_efm_image(model, train_data, M=128, n_epochs=20, batch_size=128, lr=2e-4, device='cpu'):
    """Train a UNet with EFM on image data."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []

    n = len(train_data)
    train_flat = train_data.view(n, -1)
    steps_per_epoch = n // batch_size

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        perm = torch.randperm(n)
        for i in range(steps_per_epoch):
            idx = perm[i * batch_size:(i + 1) * batch_size]
            x1 = train_data[idx].to(device)

            t = torch.rand(batch_size, 1, device=device)
            x0 = torch.randn_like(x1)
            t_expand = t[:, :, None, None]
            x_t = (1.0 - t_expand) * x0 + t_expand * x1

            x_t_flat = x_t.view(batch_size, -1)
            x1_flat = x1.view(batch_size, -1)
            target = compute_efm_target(x_t_flat, x1_flat, train_flat, t, M)
            target = target.view_as(x_t)

            pred = model(x_t, t.squeeze(-1))
            loss = ((pred - target) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg = epoch_loss / steps_per_epoch
        losses.append(avg)
        print(f'  Epoch {epoch+1}/{n_epochs} — loss: {avg:.4f}')
    return losses

## 4.3 Train Vanilla CFM

In [ ]:
N_EPOCHS = 25

print('=== Training CFM ===')
model_cfm = SmallUNet(in_channels=1, base_channels=32, time_emb_dim=64)
losses_cfm = train_cfm_image(model_cfm, all_train_images, n_epochs=N_EPOCHS,
                              batch_size=BATCH_SIZE, lr=2e-4, device=device)

## 4.4 Train EFM for various M

In [ ]:
M_values = [128, 256, 512]
models_efm = {}
losses_efm = {}

for M in M_values:
    print(f'\n=== Training EFM M={M} ===')
    model_e = SmallUNet(in_channels=1, base_channels=32, time_emb_dim=64)
    l = train_efm_image(model_e, all_train_images, M=M, n_epochs=N_EPOCHS,
                        batch_size=BATCH_SIZE, lr=2e-4, device=device)
    models_efm[M] = model_e
    losses_efm[M] = l

## 4.5 Training loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
epochs = np.arange(1, N_EPOCHS + 1)

ax.plot(epochs, losses_cfm, 'o-', linewidth=2, label='CFM (M=1)', color='steelblue')
efm_colors = ['seagreen', 'coral', 'orchid']
for i, M in enumerate(M_values):
    ax.plot(epochs, losses_efm[M], 'o-', linewidth=2, label=f'EFM M={M}', color=efm_colors[i])

ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title(f'CFM vs EFM — {DATASET}')
ax.legend()
plt.tight_layout()
plt.savefig('fig4_loss_curves.pdf', bbox_inches='tight')
plt.show()

## 4.6 Generate and visualize samples

In [ ]:
def show_samples(model, title, n=64, n_steps=100):
    samples = ode_sample(model, n, (1, IMAGE_SIZE, IMAGE_SIZE), n_steps=n_steps, device=device)
    samples = samples.clamp(-1, 1) * 0.5 + 0.5
    grid = make_grid(samples.cpu(), nrow=8, padding=1).permute(1, 2, 0)
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()
    return samples

print('--- CFM samples ---')
samples_cfm = show_samples(model_cfm, 'CFM (M=1)')

for M in M_values:
    print(f'--- EFM M={M} samples ---')
    show_samples(models_efm[M], f'EFM M={M}')

## 4.7 Nearest-neighbor distance (memorization metric)

In [ ]:
n_gen = 500

def compute_nn_dist(model, train_flat, n_gen=500):
    gen = ode_sample(model, n_gen, (1, IMAGE_SIZE, IMAGE_SIZE), n_steps=100, device=device)
    gen_flat = gen.view(n_gen, -1).cpu()
    return nearest_neighbor_distance(gen_flat, train_flat.cpu()).mean().item()

nn_cfm = compute_nn_dist(model_cfm, train_flat)
print(f'CFM NN dist:  {nn_cfm:.4f}')

nn_efm = {}
for M in M_values:
    nn_efm[M] = compute_nn_dist(models_efm[M], train_flat)
    print(f'EFM M={M} NN dist: {nn_efm[M]:.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
labels = ['CFM'] + [f'EFM M={M}' for M in M_values]
values = [nn_cfm] + [nn_efm[M] for M in M_values]
colors = ['steelblue'] + efm_colors[:len(M_values)]

ax.bar(range(len(labels)), values, color=colors, edgecolor='white')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel('Mean NN distance')
ax.set_title(f'Memorization metric — {DATASET}')
plt.tight_layout()
plt.savefig('fig4_nn_distance.pdf', bbox_inches='tight')
plt.show()

## 4.8 Loss curves per epoch — train vs test proximity

We also compute NN distance at several training checkpoints to see how generalization evolves.

In [ ]:
# Retrain CFM with periodic NN distance evaluation
print('=== Retraining CFM with NN checkpoints ===')
model_ckpt = SmallUNet(in_channels=1, base_channels=32, time_emb_dim=64).to(device)
optimizer = torch.optim.Adam(model_ckpt.parameters(), lr=2e-4)
model_ckpt.train()

n_total = len(all_train_images)
steps_per_epoch = n_total // BATCH_SIZE
nn_per_epoch = []

for epoch in range(N_EPOCHS):
    perm = torch.randperm(n_total)
    for i in range(steps_per_epoch):
        idx = perm[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        x1 = all_train_images[idx]
        t = torch.rand(BATCH_SIZE, device=device)
        x0 = torch.randn_like(x1)
        x_t = (1.0 - t[:, None, None, None]) * x0 + t[:, None, None, None] * x1
        pred = model_ckpt(x_t, t)
        loss = ((pred - (x1 - x0)) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ckpt.parameters(), 1.0)
        optimizer.step()

    nn_d = compute_nn_dist(model_ckpt, train_flat, n_gen=200)
    nn_per_epoch.append(nn_d)
    print(f'  Epoch {epoch+1} — NN dist: {nn_d:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, N_EPOCHS + 1), nn_per_epoch, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean NN distance')
ax.set_title('Memorization over training')
plt.tight_layout()
plt.savefig('fig4_memorization_over_training.pdf', bbox_inches='tight')
plt.show()